# Walk Forward: A Realistic Approach to Backtesting

In [32]:
???

Object `?` not found.


![](<src/10_Table_Validation Methods.png>)

## Load the data

In [33]:
import pandas as pd

df = pd.read_excel('data/Microsoft_LinkedIn_Processed.xlsx', parse_dates=['Date'], index_col=0)
df = df.drop(columns='change_tomorrow_direction')
df

,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,
2016-12-08,61.009998,61.580002,60.840000,61.299999,21220800,1.549141
2016-12-09,61.970001,61.990002,61.130001,61.180000,27349400,0.321694
2016-12-12,62.169998,62.299999,61.720001,61.820000,20198100,1.286125
2016-12-13,62.980000,63.419998,62.240002,62.500000,35718900,-0.478620
2016-12-14,62.680000,63.450001,62.529999,63.000000,30352700,-0.159793
...,...,...,...,...,...,...
2026-05-14,409.429993,411.839996,400.880005,404.480011,27077500,2.960282
2026-05-15,421.920013,428.170013,412.910004,414.269989,50771200,0.382489
2026-05-18,423.540009,425.119995,415.609985,416.619995,32564100,-1.466148


## Walk Forward Validation

### How `TimeSeriesSplit` works

In [34]:
from sklearn.model_selection import TimeSeriesSplit

In [35]:
ts=TimeSeriesSplit(test_size=200)

In [36]:
splits= ts.split(X=df)

In [37]:
split1=next(splits)

In [38]:
split1

(array([   0,    1,    2, ..., 1371, 1372, 1373]),
 array([1374, 1375, 1376, 1377, 1378, 1379, 1380, 1381, 1382, 1383, 1384,
        1385, 1386, 1387, 1388, 1389, 1390, 1391, 1392, 1393, 1394, 1395,
        1396, 1397, 1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405, 1406,
        1407, 1408, 1409, 1410, 1411, 1412, 1413, 1414, 1415, 1416, 1417,
        1418, 1419, 1420, 1421, 1422, 1423, 1424, 1425, 1426, 1427, 1428,
        1429, 1430, 1431, 1432, 1433, 1434, 1435, 1436, 1437, 1438, 1439,
        1440, 1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450,
        1451, 1452, 1453, 1454, 1455, 1456, 1457, 1458, 1459, 1460, 1461,
        1462, 1463, 1464, 1465, 1466, 1467, 1468, 1469, 1470, 1471, 1472,
        1473, 1474, 1475, 1476, 1477, 1478, 1479, 1480, 1481, 1482, 1483,
        1484, 1485, 1486, 1487, 1488, 1489, 1490, 1491, 1492, 1493, 1494,
        1495, 1496, 1497, 1498, 1499, 1500, 1501, 1502, 1503, 1504, 1505,
        1506, 1507, 1508, 1509, 1510, 1511, 1512, 1513, 1514,

In [39]:
split2=next(splits)

In [40]:
split2

(array([   0,    1,    2, ..., 1571, 1572, 1573]),
 array([1574, 1575, 1576, 1577, 1578, 1579, 1580, 1581, 1582, 1583, 1584,
        1585, 1586, 1587, 1588, 1589, 1590, 1591, 1592, 1593, 1594, 1595,
        1596, 1597, 1598, 1599, 1600, 1601, 1602, 1603, 1604, 1605, 1606,
        1607, 1608, 1609, 1610, 1611, 1612, 1613, 1614, 1615, 1616, 1617,
        1618, 1619, 1620, 1621, 1622, 1623, 1624, 1625, 1626, 1627, 1628,
        1629, 1630, 1631, 1632, 1633, 1634, 1635, 1636, 1637, 1638, 1639,
        1640, 1641, 1642, 1643, 1644, 1645, 1646, 1647, 1648, 1649, 1650,
        1651, 1652, 1653, 1654, 1655, 1656, 1657, 1658, 1659, 1660, 1661,
        1662, 1663, 1664, 1665, 1666, 1667, 1668, 1669, 1670, 1671, 1672,
        1673, 1674, 1675, 1676, 1677, 1678, 1679, 1680, 1681, 1682, 1683,
        1684, 1685, 1686, 1687, 1688, 1689, 1690, 1691, 1692, 1693, 1694,
        1695, 1696, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1705,
        1706, 1707, 1708, 1709, 1710, 1711, 1712, 1713, 1714,

In [41]:
list_df_train = []
list_df_test = []

for index_train, index_test in ts.split(df):
    list_df_train.append(df.iloc[index_train])
    list_df_test.append(df.iloc[index_test])


## Machine Learning Model

### Separate the data

1. Target: which variable do you want to predict?
2. Explanatory: which variables will you use to calculate the prediction?

In [42]:
y = df.change_tomorrow
X = df[['Open','High','Low','Close','Volume']]

In [43]:
list_df_train = []
list_df_test = []

for index_train, index_test in ts.split(df):
    X_train, y_train = X.iloc[index_train],y.iloc[index_train]
    X_test, y_test = X.iloc[index_test],y.iloc[index_test]

### Simulate one computation of the ML model

- Compute the model
- Calculate predictions on the test set
- Evaluate how good the model is

In [44]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

model_dt = DecisionTreeRegressor(max_depth=15, random_state=42)
model_dt.fit(X_train, y_train)

y_pred = model_dt.predict(X_test)
error_mse = mean_squared_error(y_test, y_pred)
error_mse

5.647008873478329

### Add the procedure inside the for loop

In [45]:
model_dt = DecisionTreeRegressor(max_depth=15, random_state=42)

error_mse_list = []

for index_train, index_test in ts.split(df):
    X_train, y_train= X.iloc[index_train], y.iloc[index_train]
    X_test, y_test = X.iloc[index_test], y.iloc[index_test]

    model_dt.fit(X_train,y_train)

    y_pred=model_dt.predict(X_test)
    error_mse= mean_squared_error(y_test, y_pred)

    error_mse_list.append(error_mse)

In [46]:
error_mse_list

[5.284779078562463,
 3.2038662400286193,
 1.6671576500826517,
 4.2900795443712365,
 5.647008873478329]

In [47]:
import numpy as np

In [48]:
np.mean(error_mse_list)

np.float64(4.01857827730466)

## Anchored Walk Forward evaluation in backtesting

![](<src/10_Table_Validation Methods.png>)

### Create a new strategy

In [49]:
from backtesting import Backtest, Strategy

In [50]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5

    n_train= 600
    coef_retrain = 200
    
    def init(self):
        self.model = DecisionTreeRegressor(max_depth=15, random_state=42)
        self.already_bought = False
        
        X_train =self.data.df.iloc[:self.n_train,:-1]
        y_train = self.data.df.iloc[:self.n_train,-1]
        
        self.model.fit(X=X_train, y=y_train)

    def next(self):
        explanatory_today = self.data.df.iloc[[-1],:-1]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

In [51]:
class WalkForwardAnchored(Regression):
    def next(self):
        
        # we don't take any action and move on to the following day
        if len(self.data) < self.n_train:
            return
        
        # we retrain the model each x days
        if len(self.data) % self.coef_retrain== 0:
            X_train = self.data.df.iloc[:, :-1]
            y_train = self.data.df.iloc[:, -1]

            self.model.fit(X_train, y_train)

            super().next()
            
        else:
            
            super().next()

In [52]:
from backtesting import Backtest
bt = Backtest(df, WalkForwardAnchored, cash=10000, commission=.002, exclusive_orders=True)

In [54]:
import multiprocessing as mp
mp.set_start_method('fork', force= True) 

In [55]:
stats_skopt, heatmap, optimize_result = bt.optimize(
    limit_buy = range(0, 6), limit_sell = range(-6, 0),
    maximize='Return [%]',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True,
    method='skopt'
    )

dff = heatmap.reset_index()
dff = dff.sort_values('Return [%]', ascending=False)
dff

/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(-4)] before, using random point [np.int64(4), np.int64(-2)]
  warnings.warn(


,limit_buy,limit_sell,Return [%]
7,2,-6,109.190869
20,5,-5,90.888864
16,4,-5,90.888864
0,0,-6,83.436352
21,5,-4,79.046405
12,3,-5,69.831501
13,3,-4,58.981248
3,1,-5,32.728661
4,1,-4,4.313360
8,2,-5,-10.928025


## Unanchored Walk Forward

### Create a library of strategies

`strategies.py`

### Create the unanchored walk forward class

![](<src/10_Table_Validation Methods.png>)

### Import the strategy and perform the backtest

In [ ]:
%load_ext autoreload
%autoreload 2

In [59]:
import strategies

In [60]:
strategies.WalkForwardUnanchored

strategies.WalkForwardUnanchored

In [61]:
bt_unanchored = Backtest(df, strategies.WalkForwardUnanchored, cash=10000, commission=.002, exclusive_orders=True)

stats_skopt, heatmap, optimize_result = bt_unanchored.optimize(
    limit_buy = range(0, 6), limit_sell = range(-6, 0),
    maximize='Return [%]',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True,
    method='skopt'
    )

dff = heatmap.reset_index()
dff = dff.sort_values('Return [%]', ascending=False)
dff

/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(2), np.int64(-6)] before, using random point [np.int64(4), np.int64(-6)]
  warnings.warn(
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(0), np.int64(-6)] before, using random point [np.int64(2), np.int64(-5)]
  warnings.warn(
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(0), np.int64(-6)] before, using random point [np.int64(2), np.int64(-2)]
  warnings.warn(
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(0), np.int64(-6)] before, using random point [np.int64(3), np.int64(-5)]
  warning

,limit_buy,limit_sell,Return [%]
9,2,-6,123.358041
15,3,-6,104.953871
0,0,-6,95.051649
21,4,-5,42.067352
20,4,-6,42.067352
5,1,-5,24.263032
6,1,-4,10.888238
26,5,-5,4.689932
25,5,-6,4.689932
27,5,-4,4.689932


### Interpret the strategies' performance

- Both anchored and unanchored backtesting

In [62]:
bt.plot(filename='reports_backtesting/walk_forward_anchored.html')

/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%d %b'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%m/%Y'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:455: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df2 = (df.assign(_width=1).set_index('datetime')
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:659: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/neemaurassa/Library/P

GridPlot(id='p1310', ...)

In [63]:
bt_unanchored.plot(filename='reports_backtesting/walk_forward_unanchored.html')

/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%d %b'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%m/%Y'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:455: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df2 = (df.assign(_width=1).set_index('datetime')
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:659: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/neemaurassa/Library/P

GridPlot(id='p1649', ...)

## Practice to master the knowledge

Work on the challenge with another dataset:

1. Learn the <a>mental models</a> to solve the challenge faster.
2. Complete the <a href="10C_Walk Forward Regression.ipynb">notebook</a>.